<a href="https://colab.research.google.com/github/muhammadtalhaishtiaq/project-nabula/blob/main/colab/regression/multiple_linear_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Multiple Linear Regression - From Scratch

**Project Nebula** - Don't just use ML libraries, understand how they work by building them!

---

## What You'll Learn

- How Multiple Linear Regression extends Simple Linear Regression
- The mathematics behind the Normal Equation
- How to detect multicollinearity
- Why Adjusted R² matters
- Implementing MLR from scratch using only NumPy

---

🔗 **Try it live on Project Nebula**: [project-nabula.vercel.app](https://project-nabula.vercel.app)


## 1. Introduction & Theory

### What is Multiple Linear Regression?

While Simple Linear Regression models the relationship between **one** independent variable and a dependent variable, Multiple Linear Regression (MLR) extends this to **multiple** independent variables.

**Real-world Example**: Predicting house prices based on:
- Square footage (x₁)
- Number of bedrooms (x₂)
- Age of house (x₃)
- Distance to city center (x₄)

### The Formula

$$y = \beta_0 + \beta_1x_1 + \beta_2x_2 + ... + \beta_nx_n + \epsilon$$

Where:
- **y**: Target variable (what we're predicting)
- **β₀**: Intercept (y when all x = 0)
- **β₁, β₂, ... βₙ**: Coefficients (impact of each feature)
- **x₁, x₂, ... xₙ**: Feature values
- **ε**: Error term (residual)


## 2. Mathematical Foundation

### The Normal Equation

To find the optimal coefficients, we use the **Normal Equation**:

$$\beta = (X^T X)^{-1} X^T y$$

Where:
- **X**: Feature matrix (with a column of 1s for intercept)
- **y**: Target vector
- **β**: Coefficient vector

### Adjusted R²

Regular R² always increases with more features, even if they're useless. Adjusted R² penalizes extra features:

$$R^2_{adj} = 1 - \frac{(1 - R^2)(n - 1)}{n - p - 1}$$

Where:
- **n**: Number of samples
- **p**: Number of features

### Multicollinearity Detection

When features are highly correlated, the model becomes unstable. We detect this using the **Condition Number**:

- **< 10**: No multicollinearity
- **10-30**: Moderate multicollinearity
- **> 30**: Severe multicollinearity (problematic)


## 3. Implementation from Scratch

Let's implement Multiple Linear Regression using only NumPy (no sklearn)!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(42)

print("✅ NumPy imported successfully!")
print(f"NumPy version: {np.__version__}")

In [ ]:
class MultipleLinearRegression:
    """
    Multiple Linear Regression implemented from scratch using NumPy.
    
    Features:
    - Normal Equation for coefficient calculation
    - Multicollinearity detection via condition number
    - Adjusted R² calculation
    - Standardized coefficients
    """
    
    def __init__(self):
        self.coefficients = None
        self.intercept = None
        self.is_trained = False
        self.condition_number = None
    
    def fit(self, X, y):
        """
        Train the model using Normal Equation.
        
        Parameters:
        -----------
        X : array-like, shape (n_samples, n_features)
            Training features
        y : array-like, shape (n_samples,)
            Training targets
        """
        # Add column of 1s for intercept
        X_with_intercept = np.column_stack([np.ones(len(X)), X])
        
        # Normal Equation: β = (X^T X)^(-1) X^T y
        try:
            XtX = X_with_intercept.T @ X_with_intercept
            XtX_inv = np.linalg.inv(XtX)
            Xty = X_with_intercept.T @ y
            beta = XtX_inv @ Xty
            
            # Extract intercept and coefficients
            self.intercept = beta[0]
            self.coefficients = beta[1:]
            self.is_trained = True
            
            # Calculate condition number for multicollinearity detection
            self.condition_number = np.linalg.cond(X_with_intercept)
            
        except np.linalg.LinAlgError:
            raise ValueError("Matrix is singular - features may be perfectly correlated")
    
    def predict(self, X):
        """
        Make predictions.
        
        Formula: y = intercept + sum(coefficients * features)
        """
        if not self.is_trained:
            raise ValueError("Model must be trained before prediction")
        
        return self.intercept + X @ self.coefficients
    
    def r2_score(self, X, y):
        """Calculate R² (coefficient of determination)."""
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        return 1 - (ss_res / ss_tot)
    
    def adjusted_r2_score(self, X, y):
        """Calculate Adjusted R²."""
        n = len(y)
        p = X.shape[1]
        r2 = self.r2_score(X, y)
        return 1 - ((1 - r2) * (n - 1) / (n - p - 1))
    
    def get_standardized_coefficients(self, X, y):
        """
        Calculate standardized coefficients (beta weights).
        Shows relative importance when features are on same scale.
        """
        X_std = np.std(X, axis=0)
        y_std = np.std(y)
        
        if y_std == 0:
            return np.zeros_like(self.coefficients)
        
        return self.coefficients * (X_std / y_std)
    
    def get_equation(self, feature_names):
        """Get human-readable equation."""
        equation = f"y = {self.intercept:.2f}"
        for coef, name in zip(self.coefficients, feature_names):
            sign = "+" if coef >= 0 else ""
            equation += f" {sign} {coef:.2f}*{name}"
        return equation

print("✅ MultipleLinearRegression class defined!")

## 4. Visualization

Let's create some visualizations to understand our model better.

In [ ]:
def plot_coefficients(coefficients, feature_names, title="Feature Coefficients"):
    """Plot feature coefficients as a bar chart."""
    colors = ['#3b82f6' if c >= 0 else '#ef4444' for c in coefficients]
    
    plt.figure(figsize=(10, 6))
    bars = plt.bar(feature_names, coefficients, color=colors)
    
    plt.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    plt.xlabel('Features')
    plt.ylabel('Coefficient Value')
    plt.title(title)
    plt.xticks(rotation=45, ha='right')
    
    # Add value labels on bars
    for bar, coef in zip(bars, coefficients):
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                f'{coef:.2f}',
                ha='center', va='bottom' if height >= 0 else 'top')
    
    plt.tight_layout()
    plt.show()

def plot_correlation_heatmap(X, feature_names):
    """Plot correlation matrix heatmap."""
    # Calculate correlation matrix
    n_features = X.shape[1]
    corr_matrix = np.corrcoef(X.T)
    
    plt.figure(figsize=(8, 6))
    plt.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
    plt.colorbar(label='Correlation')
    
    # Add labels
    plt.xticks(range(n_features), feature_names, rotation=45, ha='right')
    plt.yticks(range(n_features), feature_names)
    plt.title('Feature Correlation Matrix')
    
    # Add correlation values as text
    for i in range(n_features):
        for j in range(n_features):
            plt.text(j, i, f'{corr_matrix[i, j]:.2f}',
                    ha='center', va='center',
                    color='white' if abs(corr_matrix[i, j]) > 0.5 else 'black')
    
    plt.tight_layout()
    plt.show()

def plot_residuals(y_true, y_pred, title="Residual Plot"):
    """Plot residuals to check model assumptions."""
    residuals = y_true - y_pred
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Residuals vs Predicted
    ax1.scatter(y_pred, residuals, alpha=0.6)
    ax1.axhline(y=0, color='r', linestyle='--')
    ax1.set_xlabel('Predicted Values')
    ax1.set_ylabel('Residuals')
    ax1.set_title('Residuals vs Predicted')
    
    # Q-Q plot (simplified)
    ax2.hist(residuals, bins=20, edgecolor='black')
    ax2.set_xlabel('Residual Value')
    ax2.set_ylabel('Frequency')
    ax2.set_title('Distribution of Residuals')
    
    plt.tight_layout()
    plt.show()

print("✅ Visualization functions defined!")

## 5. Real Dataset Application

Now let's test our implementation on a real-world dataset!

In [ ]:
# Create synthetic house price dataset
# This mimics real-world data

np.random.seed(42)
n_samples = 100

# Features:
# x1: Square footage (1000-3000 sq ft)
# x2: Number of bedrooms (2-5)
# x3: Age of house (1-50 years)

X = np.column_stack([
    np.random.uniform(1000, 3000, n_samples),  # Size
    np.random.randint(2, 6, n_samples),        # Bedrooms
    np.random.uniform(1, 50, n_samples)        # Age
])

# Target: House price
# Formula: Price = 50000 + 100*Size + 20000*Bedrooms - 1000*Age + noise
true_coefficients = [100, 20000, -1000]
true_intercept = 50000

y = (true_intercept + 
     X[:, 0] * true_coefficients[0] + 
     X[:, 1] * true_coefficients[1] + 
     X[:, 2] * true_coefficients[2] + 
     np.random.normal(0, 15000, n_samples))  # Add noise

feature_names = ['Size_sqft', 'Bedrooms', 'Age_years']

print("✅ Dataset created!")
print(f"Samples: {len(y)}")
print(f"Features: {X.shape[1]}")
print(f"\nFeature statistics:")
for i, name in enumerate(feature_names):
    print(f"  {name}: mean={X[:, i].mean():.2f}, std={X[:, i].std():.2f}")
print(f"\nTarget (Price): mean=${y.mean():.2f}, std=${y.std():.2f}")

In [ ]:
# Split data into train/test (80/20 split)
split_idx = int(0.8 * len(X))
indices = np.random.permutation(len(X))

X_train = X[indices[:split_idx]]
X_test = X[indices[split_idx:]]
y_train = y[indices[:split_idx]]
y_test = y[indices[split_idx:]]

print(f"✅ Data split:")
print(f"  Training samples: {len(X_train)}")
print(f"  Test samples: {len(X_test)}")

In [ ]:
# Train our model
model = MultipleLinearRegression()
model.fit(X_train, y_train)

print("✅ Model trained!")
print(f"\n📊 Model Equation:")
print(f"  {model.get_equation(feature_names)}")
print(f"\n📈 Coefficients:")
for name, coef in zip(feature_names, model.coefficients):
    print(f"  {name}: {coef:.2f}")
print(f"\n🎯 Intercept: {model.intercept:.2f}")

In [ ]:
# Evaluate model
r2_train = model.r2_score(X_train, y_train)
r2_test = model.r2_score(X_test, y_test)
adj_r2 = model.adjusted_r2_score(X_test, y_test)

print("📊 Model Performance:")
print(f"  R² (training):   {r2_train:.4f}")
print(f"  R² (test):       {r2_test:.4f}")
print(f"  Adjusted R²:     {adj_r2:.4f}")
print(f"  Condition #:     {model.condition_number:.2f}")

# Multicollinearity warning
if model.condition_number > 30:
    print("\n⚠️  WARNING: Severe multicollinearity detected!")
elif model.condition_number > 10:
    print("\n⚠️  WARNING: Moderate multicollinearity detected")
else:
    print("\n✅ No multicollinearity issues")

In [ ]:
# Visualize results
print("\n📊 Feature Coefficients:")
plot_coefficients(model.coefficients, feature_names)

print("\n📊 Feature Correlations:")
plot_correlation_heatmap(X_train, feature_names)

print("\n📊 Residual Analysis:")
y_pred = model.predict(X_test)
plot_residuals(y_test, y_pred)

In [ ]:
# Standardized coefficients (relative importance)
std_coefs = model.get_standardized_coefficients(X_train, y_train)

print("📊 Standardized Coefficients (Relative Importance):")
for name, coef in zip(feature_names, std_coefs):
    print(f"  {name}: {coef:.4f}")

print("\n💡 Interpretation:")
print("  Standardized coefficients show feature importance\n  when all features are on the same scale.")
print("  Higher absolute value = more important")

## 6. Comparison with sklearn

Let's verify our implementation matches sklearn's results!

In [ ]:
# Install sklearn if not available
try:
    from sklearn.linear_model import LinearRegression
    from sklearn.metrics import r2_score
    sklearn_available = True
except ImportError:
    print("Installing scikit-learn...")
    !pip install scikit-learn -q
    from sklearn.linear_model import LinearRegression
    from sklearn.metrics import r2_score
    sklearn_available = True

# Train sklearn model
sklearn_model = LinearRegression()
sklearn_model.fit(X_train, y_train)
sklearn_pred = sklearn_model.predict(X_test)
sklearn_r2 = r2_score(y_test, sklearn_pred)

print("🔍 Comparing our implementation with sklearn:")
print(f"\nOur R²:      {r2_test:.6f}")
print(f"Sklearn R²:  {sklearn_r2:.6f}")
print(f"Difference:  {abs(r2_test - sklearn_r2):.10f}")

print(f"\nOur intercept:      {model.intercept:.4f}")
print(f"Sklearn intercept:  {sklearn_model.intercept_:.4f}")

print(f"\nOur coefficients:      {model.coefficients}")
print(f"Sklearn coefficients:  {sklearn_model.coef_}")

if abs(r2_test - sklearn_r2) < 0.0001:
    print("\n✅ MATCH! Our implementation is correct!")
else:
    print("\n⚠️  Small difference - check implementation")

## 🎉 Congratulations!

You've successfully implemented Multiple Linear Regression from scratch!

### What You Learned:
- ✅ Normal Equation for finding optimal coefficients
- ✅ Adjusted R² for model evaluation
- ✅ Multicollinearity detection with Condition Number
- ✅ Standardized coefficients for feature importance
- ✅ Visualization of results

### Next Steps:
- Try this on real datasets (housing, car prices, etc.)
- Explore regularization (Ridge, Lasso)
- Try on Project Nebula: [project-nabula.vercel.app](https://project-nabula.vercel.app)

---

**Built with ❤️ for Project Nebula**

🔗 GitHub: [github.com/muhammadtalhaishtiaq/project-nabula](https://github.com/muhammadtalhaishtiaq/project-nabula)